In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/race_driver_labels.csv")
df.head()

,raceId,driverId,constructorId,grid,positionOrder,points,statusId,year,round,circuitId,...,avgPitDuration_ms,driverName_x,constructorName_x,driverName_y,constructorName_y,driverName,constructorName,finishPosition,avgLapTime_s,constructorPoints
0,18,1,1,1,1,10.0,1,2008,1,1,...,0.0,Lewis Hamilton,McLaren,Lewis Hamilton,McLaren,Lewis Hamilton,McLaren,1,98.114069,14.0
1,18,2,2,5,2,8.0,1,2008,1,1,...,0.0,Nick Heidfeld,BMW Sauber,Nick Heidfeld,BMW Sauber,Nick Heidfeld,BMW Sauber,2,98.208517,8.0
2,18,3,3,7,3,6.0,1,2008,1,1,...,0.0,Nico Rosberg,Williams,Nico Rosberg,Williams,Nico Rosberg,Williams,3,98.254810,9.0
3,18,4,4,11,4,5.0,1,2008,1,1,...,0.0,Fernando Alonso,Renault,Fernando Alonso,Renault,Fernando Alonso,Renault,4,98.410293,5.0
4,18,5,1,3,5,4.0,1,2008,1,1,...,0.0,Heikki Kovalainen,McLaren,Heikki Kovalainen,McLaren,Heikki Kovalainen,McLaren,5,98.424655,14.0


In [18]:
df = df.sort_values(by=["driverId", "year", "round"]).reset_index(drop=True)

Sort data (CRITICAL step)
Why this matters:
Rolling features must respect time order
If data isn’t sorted, your “past” will include the future (data leakage).

Explanation:
Grouping later depends on correct ordering
raceId increases over time in your dataset → safe proxy for race order

In [19]:
df["driver_form_avg"] = (
    df.groupby(["driverId", "year"])["points"]
      .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

use rolling average finish position over last 5 races.
Key rules:
Use .shift(1) → no peeking into the current race
Use groupby(driverId)

For each row:
Average of previous 5 finishes
First race(s) → fewer data points (that’s OK)

Error fixed:
groupby().apply() returns a Series with a grouped (multi) index
df["new_column"] = ... expects a Series with the same flat index as df
Pandas refuses to guess how to align them → error.
or column-wise features, you should almost always use:
groupby().transform(...)
transform guarantees the output has the same index as the original DataFrame.

Rule of thumb:
Creating a new column → use transform
Creating a reduced table → use apply or agg

In [20]:
df = df.sort_values(by=["constructorId", "year", "round"]).reset_index(drop=True)

df["constructor_form_avg"] = (
    df.groupby(["constructorId", "year"])["constructorPoints"]
      .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

use rolling average constructor points over last 3 races.

Why constructorPoints:
Reflects car performance
Includes both drivers’ contribution
Less noisy than lap times

In [21]:
circuits = pd.read_csv("../data/raw/races.csv")
circuits.head()

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
2,3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
3,4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
4,5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


In [22]:
street_circuits = [
    "Monaco Grand Prix", "Singapore Grand Prix", "Baku Grand Prix", "Saudi Arabian Grand Prix",
    "Las Vegas Grand Prix", "Miami Grand Prix", "Australian Grand Prix"
]

circuits["circuitType"] = np.where(
    circuits["name"].isin(street_circuits),
    "street",
    "permanent"
)

In [23]:
df = df.merge(
    circuits[["circuitId", "circuitType"]],
    on="circuitId",
    how="left"
)

In [24]:
df = pd.get_dummies(df, columns=["circuitType"], drop_first=True)

In [25]:
feature_cols = [
    "qualifyingPosition",
    "pitStopCount",
    "driver_form_avg",
    "constructor_form_avg",
    "circuitType_street"
]

features_df = df[feature_cols + [
    "finishPosition",
    "avgLapTime_s",
    "constructorPoints"
]]

In [26]:
features_df.to_csv(
    "../data/processed/features_v1.csv",
    index=False
)

What has been done:
Converted raw race data into temporal performance signals
Avoided data leakage correctly
Added domain-aware features
Created a reusable ML-ready dataset
Built a clean, explainable pipeline